# 🔧 Feature Engineering — SP500 M5
## Construction des features & Labélisation (N=6 bougies / 30min)

**Basé sur** : Dingli & Sant Fournier (2017) — *Financial Time Series Forecasting: A Machine Learning Approach*

Ce notebook construit toutes les features nécessaires au modèle de classification Up/Down :

1. Imports & Paramètres
2. Chargement & Préparation des données
3. Labélisation — N=6 bougies (30min), seuil=1.2 pts
4. Features de prix de base
5. Indicateurs de tendance (MA, MACD)
6. Indicateurs de momentum (RSI, Stochastique, CCI, ROC...)
7. Indicateurs de volatilité (ATR, Bollinger)
8. Indicateurs de volume (OBV, Force Index)
9. Features temporelles (heure, session, jour)
10. Features inter-actifs (Gold, Brent, EURUSD)
11. Features laggées
12. Récapitulatif & Nettoyage final
13. Feature Scaling (Z-Score)
14. Export & Aperçu corrélations

## 0. Imports & Paramètres

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='darkgrid')

# ── Paramètres label (issus du notebook 01) ───────────────────────────────
N_LABEL   = 6      # 6 bougies = 30 minutes
SEUIL_PTS = 1.2    # spread aller-retour Admirals

# ── Paramètres indicateurs ────────────────────────────────────────────────
FAST    = 12  ; SLOW   = 26  ; SIGNAL = 9
RSI_P   = 14  ; BB_P   = 20  ; ATR_P  = 14
STOCH_P = 14  ; CCI_P  = 20  ; WILL_P = 14
ROC_P   = 10  ; MOM_P  = 10

print('✅ Imports OK')
print(f'   Label : N={N_LABEL} bougies ({N_LABEL*5}min) | Seuil={SEUIL_PTS} pts')

✅ Imports OK
   Label : N=6 bougies (30min) | Seuil=1.2 pts


## 1. Chargement & Préparation des données

In [13]:
DATA_DIR = Path('../SCR/data')

sp500  = pd.read_csv(DATA_DIR / 'sp500_m5.csv',  index_col='time', parse_dates=True)
gold   = pd.read_csv(DATA_DIR / 'or_m5.csv',     index_col='time', parse_dates=True)
brent  = pd.read_csv(DATA_DIR / 'brent_m5.csv',  index_col='time', parse_dates=True)
eurusd = pd.read_csv(DATA_DIR / 'eurusd_m5.csv', index_col='time', parse_dates=True)

# Nettoyage SP500
df = sp500.copy().sort_index()
df = df[~df.index.duplicated(keep='first')]
df = df.dropna(subset=['close'])
df.columns = [c.lower() for c in df.columns]

# Nettoyage actifs externes
for ext in [gold, brent, eurusd]:
    ext.sort_index(inplace=True)
    ext.drop_duplicates(inplace=True)

print(f'✅ SP500  : {len(df):,} bougies | {df.index.min().date()} → {df.index.max().date()}')
print(f'   Gold   : {len(gold):,} | Brent : {len(brent):,} | EURUSD : {len(eurusd):,}')
print(f'   Colonnes SP500 : {list(df.columns)}')
df.head(3)

✅ SP500  : 99,999 bougies | 2024-11-14 → 2026-04-17
   Gold   : 99,999 | Brent : 99,689 | EURUSD : 99,997
   Colonnes SP500 : ['open', 'high', 'low', 'close', 'tick_volume']


,open,high,low,close,tick_volume
time,,,,,
2024-11-14 15:20:00+00:00,5995.33,5995.71,5994.22,5995.50,105
2024-11-14 15:25:00+00:00,5995.74,5996.00,5994.49,5995.73,100
2024-11-14 15:30:00+00:00,5995.48,5995.85,5985.72,5986.59,494


In [14]:
import ta


In [15]:
import ta

feat = df.copy()

o = feat['open']
h = feat['high']
l = feat['low']
c = feat['close']
v = feat['tick_volume']

# --- OVERLAP ---
feat['sma_10']   = ta.trend.sma_indicator(c, window=10)
feat['sma_20']   = ta.trend.sma_indicator(c, window=20)
feat['sma_50']   = ta.trend.sma_indicator(c, window=50)
feat['ema_10']   = ta.trend.ema_indicator(c, window=10)
feat['ema_20']   = ta.trend.ema_indicator(c, window=20)
feat['ema_50']   = ta.trend.ema_indicator(c, window=50)

# --- MOMENTUM ---
feat['rsi_14']   = ta.momentum.rsi(c, window=14)
feat['rsi_7']    = ta.momentum.rsi(c, window=7)
macd_obj         = ta.trend.MACD(c)
feat['macd']     = macd_obj.macd()
feat['macd_sig'] = macd_obj.macd_signal()
feat['macd_dif'] = macd_obj.macd_diff()
stoch_obj        = ta.momentum.StochasticOscillator(h, l, c)
feat['stoch_k']  = stoch_obj.stoch()
feat['stoch_d']  = stoch_obj.stoch_signal()
feat['cci']      = ta.trend.cci(h, l, c, window=20)
feat['roc']      = ta.momentum.roc(c, window=10)
feat['williams'] = ta.momentum.williams_r(h, l, c, lbp=14)

# --- VOLUME ---
feat['obv']      = ta.volume.on_balance_volume(c, v)
feat['cmf']      = ta.volume.chaikin_money_flow(h, l, c, v, window=20)
feat['mfi']      = ta.volume.money_flow_index(h, l, c, v, window=14)

# --- VOLATILITE ---
bb_obj           = ta.volatility.BollingerBands(c, window=20)
feat['bb_high']  = bb_obj.bollinger_hband()
feat['bb_low']   = bb_obj.bollinger_lband()
feat['bb_width'] = bb_obj.bollinger_wband()
feat['atr']      = ta.volatility.average_true_range(h, l, c, window=14)
kelt_obj         = ta.volatility.KeltnerChannel(h, l, c, window=20)
feat['kelt_h']   = kelt_obj.keltner_channel_hband()
feat['kelt_l']   = kelt_obj.keltner_channel_lband()

# --- TRANSFORMATION EN RENDEMENTS ---
# Prix relatifs au close (écart en %)
prix_cols = ['open', 'high', 'low',
             'sma_10', 'sma_20', 'sma_50',
             'ema_10', 'ema_20', 'ema_50',
             'bb_high', 'bb_low',
             'kelt_h', 'kelt_l']

for col in prix_cols:
    feat[col] = (feat[col] - feat['close']) / feat['close']

# Close devient rendement période à période
feat['close'] = feat['close'].pct_change()

# OBV en rendement aussi
feat['obv'] = feat['obv'].pct_change()

print(f"Shape : {feat.shape}")
print(f"Colonnes : {list(feat.columns)}")
print(f"NaN par colonne :\n{feat.isna().sum()}")

Shape : (99999, 30)
Colonnes : ['open', 'high', 'low', 'close', 'tick_volume', 'sma_10', 'sma_20', 'sma_50', 'ema_10', 'ema_20', 'ema_50', 'rsi_14', 'rsi_7', 'macd', 'macd_sig', 'macd_dif', 'stoch_k', 'stoch_d', 'cci', 'roc', 'williams', 'obv', 'cmf', 'mfi', 'bb_high', 'bb_low', 'bb_width', 'atr', 'kelt_h', 'kelt_l']
NaN par colonne :
open            0
high            0
low             0
close           1
tick_volume     0
sma_10          9
sma_20         19
sma_50         49
ema_10          9
ema_20         19
ema_50         49
rsi_14         13
rsi_7           6
macd           25
macd_sig       33
macd_dif       33
stoch_k        13
stoch_d        15
cci            19
roc            10
williams       13
obv             1
cmf            19
mfi            13
bb_high        19
bb_low         19
bb_width       19
atr             0
kelt_h          0
kelt_l          0
dtype: int64


In [16]:
feat = feat.dropna()

print(f"Shape après dropna : {feat.shape}")
print(f"NaN restants : {feat.isna().sum().sum()}")
feat.head(3)

Shape après dropna : (99950, 30)
NaN restants : 0


,open,high,low,close,tick_volume,sma_10,sma_20,sma_50,ema_10,ema_20,...,williams,obv,cmf,mfi,bb_high,bb_low,bb_width,atr,kelt_h,kelt_l
time,,,,,,,,,,,,,,,,,,,,,
2024-11-14 19:25:00+00:00,0.000134,0.000298,-0.000159,-0.000134,341,0.001126,0.001908,0.002766,0.000982,0.001613,...,-95.722647,0.096765,-0.193897,33.212201,0.003933,-0.000118,0.404338,4.709337,0.002734,0.001215
2024-11-14 19:30:00+00:00,0.000549,0.000549,-0.001037,-0.000506,473,0.001295,0.002231,0.003156,0.001218,0.001918,...,-79.711097,0.122380,-0.138056,25.234805,0.004427,0.000036,0.438102,5.047956,0.003091,0.001476
2024-11-14 19:35:00+00:00,-0.001109,0.000335,-0.001109,0.001069,366,0.000085,0.001072,0.001988,0.000122,0.000768,...,-58.798424,-0.084371,-0.077488,26.508646,0.003300,-0.001157,0.445228,5.303102,0.001916,0.000283


In [17]:
import numpy as np

seuil = 2.0
max_bougies = 50

closes = df['close'].values
resultats = []

for i in range(len(closes) - max_bougies):
    ref = closes[i]
    for j in range(1, max_bougies + 1):
        diff = closes[i + j] - ref
        if abs(diff) >= seuil:
            resultats.append({
                'bougies'   : j,
                'direction' : 'Up' if diff > 0 else 'Down',
                'points'    : diff
            })
            break
    else:
        resultats.append({'bougies': None, 'direction': 'Never', 'points': None})

res_df = pd.DataFrame(resultats)
atteint = res_df[res_df['bougies'].notna()]
print(f"Médiane : {atteint['bougies'].median()} bougies")
print(atteint['bougies'].describe(percentiles=[.25, .5, .75, .90]))

Médiane : 2.0 bougies
count    99836.000000
mean         3.851376
std          4.665188
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
90%          9.000000
max         50.000000
Name: bougies, dtype: float64


In [18]:
seuil = 2.0
N = 9

diff = df['close'].reindex(feat.index).shift(-N) - df['close'].reindex(feat.index)

feat['target'] = 0
feat['target'] = feat['target'].mask(diff >  seuil,  1)
feat['target'] = feat['target'].mask(diff < -seuil,  2)

feat = feat.iloc[:-N]

print(f"Distribution de la cible :")
print(feat['target'].value_counts().sort_index())
print(f"\nEn pourcentage :")
print(feat['target'].value_counts(normalize=True).sort_index().mul(100).round(1))

Distribution de la cible :
target
0    30031
1    36832
2    33078
Name: count, dtype: int64

En pourcentage :
target
0    30.0
1    36.9
2    33.1
Name: proportion, dtype: float64


In [19]:
from sklearn.preprocessing import StandardScaler

print(f"Cible actuelle : {feat['target'].value_counts().sort_index().to_dict()}")

feature_cols = [c for c in feat.columns if c != 'target']

split = int(len(feat) * 0.80)

train = feat.iloc[:split]
test  = feat.iloc[split:]

print(f"\nTrain : {len(train):,} | {train.index.min().date()} → {train.index.max().date()}")
print(f"Test  : {len(test):,}  | {test.index.min().date()} → {test.index.max().date()}")

scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_cols])
X_test  = scaler.transform(test[feature_cols])

y_train = train['target'].values
y_test  = test['target'].values

print(f"\nX_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"\nDistribution y_train : {pd.Series(y_train).value_counts().sort_index().to_dict()}")
print(f"Distribution y_test  : {pd.Series(y_test).value_counts().sort_index().to_dict()}")

Cible actuelle : {0: 30031, 1: 36832, 2: 33078}

Train : 79,952 | 2024-11-14 → 2026-01-06
Test  : 19,989  | 2026-01-06 → 2026-04-17

X_train : (79952, 30)
X_test  : (19989, 30)

Distribution y_train : {0: 25357, 1: 28912, 2: 25683}
Distribution y_test  : {0: 4674, 1: 7920, 2: 7395}


In [20]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X_train, y_train)

scores_df = pd.DataFrame({
    'feature' : feature_cols,
    'score'   : selector.scores_,
    'pvalue'  : selector.pvalues_
}).sort_values('score', ascending=False)

print(scores_df.to_string(index=False))

    feature       score        pvalue
tick_volume 3310.202003  0.000000e+00
        atr 2586.442292  0.000000e+00
   bb_width 1504.304941  0.000000e+00
    bb_high 1043.757620  0.000000e+00
       high 1008.963957  0.000000e+00
        low  937.690267  0.000000e+00
     bb_low  667.309852 3.831534e-288
     kelt_h  481.021140 2.198249e-208
     rsi_14  160.422528  2.943111e-70
     kelt_l  148.238027  5.496893e-65
      rsi_7  102.917274  2.296329e-45
        cci  100.024856  4.111637e-44
        mfi   84.375596  2.481776e-37
    stoch_d   81.945386  2.805423e-36
    stoch_k   77.442896  2.509080e-34
   williams   77.442896  2.509080e-34
     ema_50   74.913532  3.132664e-33
   macd_sig   69.689204  5.763874e-31
       macd   67.893548  3.461132e-30
     sma_50   57.259198  1.413967e-25
        cmf   48.027783  1.426656e-21
     ema_20   38.499726  1.940117e-17
     sma_20   30.572626  5.340132e-14
     ema_10   22.905827  1.134945e-10
        roc   20.834282  8.997947e-10
     sma_10 

In [21]:
selected_features = scores_df[scores_df['pvalue'] < 0.05]['feature'].tolist()
print(f"Features sélectionnées ({len(selected_features)}) : {selected_features}")

selector_final = SelectKBest(score_func=f_classif, k=len(selected_features))
selector_final.fit(X_train, y_train)

X_train_sel = selector_final.transform(X_train)
X_test_sel  = selector_final.transform(X_test)

print(f"\nX_train_sel : {X_train_sel.shape}")
print(f"X_test_sel  : {X_test_sel.shape}")

Features sélectionnées (28) : ['tick_volume', 'atr', 'bb_width', 'bb_high', 'high', 'low', 'bb_low', 'kelt_h', 'rsi_14', 'kelt_l', 'rsi_7', 'cci', 'mfi', 'stoch_d', 'stoch_k', 'williams', 'ema_50', 'macd_sig', 'macd', 'sma_50', 'cmf', 'ema_20', 'sma_20', 'ema_10', 'roc', 'sma_10', 'close', 'open']

X_train_sel : (79952, 28)
X_test_sel  : (19989, 28)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

classifiers = {
    'KNN'               : KNeighborsClassifier(),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'NaiveBayes'        : GaussianNB(),
    'SVC'               : SVC(),
    'DecisionTree'      : DecisionTreeClassifier(),
    'RandomForest'      : RandomForestClassifier(n_jobs=-1),
    'MLP'               : MLPClassifier(max_iter=1000),
    'AdaBoost'          : AdaBoostClassifier(),
    'QDA'               : QuadraticDiscriminantAnalysis()
}

results = []
for name, clf in classifiers.items():
    start = time.time()
    clf.fit(X_train_sel, y_train)
    elapsed = time.time() - start

    y_pred = clf.predict(X_test_sel)

    results.append({
        'Classifier' : name,
        'Accuracy'   : accuracy_score(y_test, y_pred),
        'Precision'  : precision_score(y_test, y_pred, average='weighted'),
        'Recall'     : recall_score(y_test, y_pred,    average='weighted'),
        'F1'         : f1_score(y_test, y_pred,        average='weighted'),
        'Time (s)'   : round(elapsed, 2)
    })
    print(f"✅ {name} — Accuracy: {accuracy_score(y_test, y_pred):.4f} | F1: {f1_score(y_test, y_pred, average='weighted'):.4f} ({elapsed:.1f}s)")

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print(f"\n{results_df.to_string(index=False)}")

✅ KNN — Accuracy: 0.3798 | F1: 0.3758 (0.0s)
✅ LogisticRegression — Accuracy: 0.4105 | F1: 0.3205 (1.7s)
✅ NaiveBayes — Accuracy: 0.3349 | F1: 0.2979 (0.0s)
